In [1]:
import sys
sys.path.insert(0, r"C:\Users\mjbou\governance-framework\src")
from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io

log = load_log()
print(f"Log loaded. Rows: {len(log)}")

Log loaded. Rows: 4


## FSI Pipeline

**Source:** Fragile States Index (Fund for Peace)
**Access:** Automated — scrapes download page to find latest Excel file
**Download instructions:** See `docs/instructions_data_maintenance.md` — FSI section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| C1: Security Apparatus | State capacity | Primary tier 1 |
| C2: Factionalized Elites | Political settlement | Primary tier 1 |
| C3: Group Grievance | Political settlement | Primary tier 1 |
| P2: Public Services | Service delivery | Primary tier 2 |

In [3]:
import requests
import re
import io
from datetime import datetime

FSI_DOWNLOAD_PAGE = "https://fragilestatesindex.org/excel/"

# Add browser headers to avoid 403 block
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def get_fsi_data():
    """
    Scrape FSI download page to find all Excel links,
    select the most recent year, and download it.
    Uses browser User-Agent to avoid 403 blocks.
    """
    response = requests.get(FSI_DOWNLOAD_PAGE, headers=HEADERS, timeout=30)
    if response.status_code != 200:
        print(f"Failed to reach FSI page: {response.status_code}")
        return None, None, None

    matches = re.findall(
        r'href="(https://fragilestatesindex\.org/wp-content/uploads/[^"]+\.xlsx)"',
        response.text
    )

    if not matches:
        print("No Excel links found on FSI page")
        return None, None, None

    def extract_year(url):
        year_match = re.search(r'fsi[-_](\d{4})', url.lower())
        return int(year_match.group(1)) if year_match else 0

    urls_with_years = [(url, extract_year(url)) for url in matches]
    urls_with_years = [(url, yr) for url, yr in urls_with_years if yr > 0]
    urls_with_years.sort(key=lambda x: x[1], reverse=True)

    print(f"Found {len(urls_with_years)} FSI Excel files on page")
    print(f"Most recent: {urls_with_years[0]}")

    best_url, best_year = urls_with_years[0]
    file_response = requests.get(best_url, headers=HEADERS, timeout=60)

    if file_response.status_code == 200 and 'text/html' not in file_response.headers.get('Content-Type', ''):
        return best_url, best_year, file_response.content

    print(f"Download failed for {best_url}: {file_response.status_code}")
    return None, None, None

print("Searching for latest FSI data file...")
FSI_URL, FSI_YEAR, FSI_CONTENT = get_fsi_data()

if FSI_CONTENT:
    print(f"Downloaded FSI {FSI_YEAR}. Size: {len(FSI_CONTENT)/1024:.1f} KB")
    xl = pd.ExcelFile(io.BytesIO(FSI_CONTENT), engine='openpyxl')
    print(f"Sheets: {xl.sheet_names}")
else:
    print("⚠️ FSI download failed. See docs/instructions_data_maintenance.md — FSI section.")

Searching for latest FSI data file...
Found 35 FSI Excel files on page
Most recent: ('https://fragilestatesindex.org/wp-content/uploads/2023/06/FSI-2023-DOWNLOAD.xlsx', 2023)
Downloaded FSI 2023. Size: 26.3 KB
Sheets: ['Sheet1']


In [4]:
# Load and inspect
df_raw = xl.parse('Sheet1')
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(df_raw.head(3))

Shape: (179, 16)
Columns: ['Country', 'Year', 'Rank', 'Total', 'S1: Demographic Pressures', 'S2: Refugees and IDPs', 'C3: Group Grievance', 'E3: Human Flight and Brain Drain', 'E2: Economic Inequality', 'E1: Economy', 'P1: State Legitimacy', 'P2: Public Services', 'P3: Human Rights', 'C1: Security Apparatus', 'C2: Factionalized Elites', 'X1: External Intervention']
       Country  Year Rank  Total  S1: Demographic Pressures  \
0      Somalia  2023  1st  111.9                       10.0   
1        Yemen  2023  2nd  108.9                        9.6   
2  South Sudan  2023  3rd  108.5                        9.7   

   S2: Refugees and IDPs  C3: Group Grievance  \
0                    9.0                  8.7   
1                    9.6                  8.8   
2                   10.0                  8.6   

   E3: Human Flight and Brain Drain  E2: Economic Inequality  E1: Economy  \
0                               8.6                      9.1          9.5   
1                           

In [6]:
import requests
import re
import io
from datetime import datetime

FSI_DOWNLOAD_PAGE = "https://fragilestatesindex.org/excel/"
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def get_all_fsi_urls():
    """Scrape FSI download page and return deduplicated Excel URLs by year."""
    response = requests.get(FSI_DOWNLOAD_PAGE, headers=HEADERS, timeout=30)
    if response.status_code != 200:
        print(f"Failed to reach FSI page: {response.status_code}")
        return []

    matches = re.findall(
        r'href="(https://fragilestatesindex\.org/wp-content/uploads/[^"]+\.xlsx)"',
        response.text
    )

    def extract_year(url):
        year_match = re.search(r'fsi[-_](\d{4})', url.lower())
        return int(year_match.group(1)) if year_match else 0

    # Deduplicate — keep first URL found per year
    seen_years = {}
    for url in matches:
        year = extract_year(url)
        if year >= FRAMEWORK_START_YEAR and year not in seen_years:
            seen_years[year] = url

    return sorted(seen_years.items())

def download_fsi_year(url):
    """Download a single FSI Excel file and return its content."""
    response = requests.get(url, headers=HEADERS, timeout=60)
    if response.status_code == 200 and 'text/html' not in response.headers.get('Content-Type', ''):
        return response.content
    return None

print("Fetching FSI file list...")
fsi_urls = get_all_fsi_urls()
print(f"Found {len(fsi_urls)} unique years from {FRAMEWORK_START_YEAR} onwards")
print(f"Years: {[yr for yr, _ in fsi_urls]}")

# Download and stack all years
frames = []
for year, url in fsi_urls:
    content = download_fsi_year(url)
    if content:
        xl_yr = pd.ExcelFile(io.BytesIO(content), engine='openpyxl')
        df_yr = xl_yr.parse(xl_yr.sheet_names[0])
        df_yr['Year'] = year
        frames.append(df_yr)
        print(f"  Downloaded {year}: {len(df_yr)} rows")
    else:
        print(f"  ⚠️ Failed: {year}")

fsi_all = pd.concat(frames, ignore_index=True)
print(f"\nCombined shape: {fsi_all.shape}")
print(f"Years: {sorted(fsi_all['Year'].unique())}")

Fetching FSI file list...
Found 18 unique years from 1990 onwards
Years: [2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
  Downloaded 2006: 146 rows
  Downloaded 2007: 177 rows
  Downloaded 2008: 177 rows
  Downloaded 2009: 177 rows
  Downloaded 2010: 177 rows
  Downloaded 2011: 177 rows
  Downloaded 2012: 178 rows
  Downloaded 2013: 178 rows
  Downloaded 2014: 178 rows
  Downloaded 2015: 178 rows
  Downloaded 2016: 178 rows
  Downloaded 2017: 178 rows
  Downloaded 2018: 178 rows
  Downloaded 2019: 178 rows
  Downloaded 2020: 178 rows
  Downloaded 2021: 179 rows
  Downloaded 2022: 179 rows
  Downloaded 2023: 179 rows

Combined shape: (3170, 17)
Years: [np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int

In [7]:
# Framework indicators from FSI
# C1: Security Apparatus → State capacity (P1)
# C2: Factionalized Elites → Political settlement (P1)
# C3: Group Grievance → Political settlement (P1)  
# P2: Public Services → Service delivery (P2)

# Standardise column names across years — some years have slightly different names
# Check what columns we actually have
print("Columns in combined dataset:")
print(list(fsi_all.columns))

Columns in combined dataset:
['Country', 'Year', 'Rank', 'Total', 'C1: Security Apparatus', 'C2: Factionalized Elites', 'C3: Group Grievance', 'E1: Economy', 'E2: Economic Inequality', 'E3: Human Flight and Brain Drain', 'P1: State Legitimacy', 'P2: Public Services', 'P3: Human Rights', 'S1: Demographic Pressures', 'S2: Refugees and IDPs', 'X1: External Intervention', 'Change from Previous Year']


In [8]:
# Filter to framework indicators plus identifiers
KEEP_COLS = [
    'Country', 'Year',
    'C1: Security Apparatus',
    'C2: Factionalized Elites',
    'C3: Group Grievance',
    'P2: Public Services',
]

fsi = fsi_all[KEEP_COLS].copy()

# Rename columns
fsi = fsi.rename(columns={
    'Country':                  'country_name',
    'Year':                     'year',
    'C1: Security Apparatus':   'fsi_c1_security_apparatus',
    'C2: Factionalized Elites': 'fsi_c2_factionalized_elites',
    'C3: Group Grievance':      'fsi_c3_group_grievance',
    'P2: Public Services':      'fsi_p2_public_services',
})

# Clean year column
fsi['year'] = fsi['year'].astype(int)

# Sort
fsi = fsi.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {fsi.shape}")
print(f"Years: {sorted(fsi['year'].unique())}")
print(f"Countries: {fsi['country_name'].nunique()}")
print(f"\nMissing values (%):")
missing_pct = (fsi.isnull().sum() / len(fsi) * 100).round(1)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(fsi.head())

Shape: (3170, 6)
Years: [np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Countries: 231

Missing values (%):
Series([], dtype: float64)
  country_name  year  fsi_c1_security_apparatus  fsi_c2_factionalized_elites  \
0  Afghanistan  2006                        8.2                          8.0   
1  Afghanistan  2007                        9.0                          8.5   
2  Afghanistan  2008                        9.6                          8.8   
3  Afghanistan  2009                        9.9                          9.1   
4  Afghanistan  2010                        9.7                          9.4   

   fsi_c3_group_grievance  fsi_p2_public_services  
0                     9.1                     8.0  
1                     9.1              

In [9]:
# Save to processed
output_path = os.path.join(PROCESSED_DIR, "fsi_clean.csv")
fsi.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {fsi.shape}")

# Derive metadata from data
latest_year = str(int(fsi['year'].max()))
data_as_of_date = latest_year

# Update download log
update_entry(
    "FSI",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of_date,
    local_filename="fsi_clean.csv",
    latest_available_version=latest_year,
    notes=f"Indicators C1, C2, C3, P2. All years scraped and stacked automatically. Latest on page: {latest_year}. 2024/2025 not yet available."
)

print_entry("FSI")

Written: C:\Users\mjbou\governance-framework\data\processed\fsi_clean.csv
Shape: (3170, 6)
[download_log] Updated entry for FSI
  source_id: FSI
  last_attempted_date: 2026-05-21
  last_successful_download_date: 2026-05-21
  data_as_of_date: 2023
  local_filename: fsi_clean.csv
  latest_available_version: 2023
  no_update_reason: nan
  notes: Indicators C1, C2, C3, P2. All years scraped and stacked automatically. Latest on page: 2023. 2024/2025 not yet available.
